# UD2.02. Gestión de secretos: dónde va una clave de API

**Módulo 5073 · Programación de Inteligencia Artificial · UD2**

Este cuaderno va **antes** de la primera llamada a un servicio de pago. No al final de la unidad,
no "cuando ya funcione": antes.

El motivo es que la costumbre se coge la primera vez. Quien escribe la clave en la celda para
"probar rápido" acaba subiéndola, y entonces ya da igual lo bien que esté el resto del proyecto.

> **Una clave de API nunca se escribe en el código. Nunca.**

Al terminar tendrás montado el `.env` de la unidad, sabrás leer la clave en local, en Colab y en
Streamlit con el mismo código, y sabrás qué hacer el día que se te escape una.

In [ ]:
!pip install -q python-dotenv requests

## 1. Por qué "borrarla luego" no sirve

Una clave subida a un repositorio está comprometida en el momento en que se sube. Aunque el
repositorio sea privado. Aunque la borres en el commit siguiente.

Dos razones:

1. **Git guarda el historial.** El commit anterior sigue ahí, y `git log -p` lo enseña. Borrar la
   línea en un commit nuevo no borra nada, solo añade una línea más.
2. **Hay rastreadores automáticos** recorriendo GitHub que detectan patrones de clave en minutos.
   Los repositorios privados se vuelven públicos por error más a menudo de lo que parece, y una
   clave filtrada de un servicio de pago se usa para minar criptomonedas o para revender
   llamadas.

Compruébalo tú, sin necesidad de subir nada:

In [ ]:
# Simulación local: una clave escrita en un fichero, "borrada" después.
# Se ve que el historial la conserva.
import subprocess, tempfile, os, pathlib

tmp = tempfile.mkdtemp()
ejecuta = lambda *args: subprocess.run(args, cwd=tmp, capture_output=True, text=True)

ejecuta("git", "init", "-q")
ejecuta("git", "config", "user.email", "prueba@ejemplo.org")
ejecuta("git", "config", "user.name", "Prueba")

pathlib.Path(tmp, "app.py").write_text('CLAVE = "sk-clave-secreta-123456"\n')
ejecuta("git", "add", "."); ejecuta("git", "commit", "-qm", "primera version")

pathlib.Path(tmp, "app.py").write_text('CLAVE = os.getenv("CLAVE")\n')
ejecuta("git", "add", "."); ejecuta("git", "commit", "-qm", "quito la clave")

print("Contenido actual del fichero:")
print("  ", pathlib.Path(tmp, "app.py").read_text().strip())
print()
print("Y esto es lo que sigue estando en el historial:")
for linea in ejecuta("git", "log", "-p", "--", "app.py").stdout.split("\n"):
    if "clave-secreta" in linea:
        print("  ", linea)

Ahí está: el fichero está limpio y la clave sigue en el repositorio.

Limpiar el historial de verdad exige reescribirlo, y si el repositorio es compartido eso rompe el
trabajo de las demás personas. Por eso **el primer paso nunca es limpiar: es rotar la clave**.

## 2. Dónde va entonces: el fichero `.env`

Un fichero de texto, en la raíz del proyecto, **listado en `.gitignore`**:

```bash
# .env  -> NO se sube nunca
AZURE_LANGUAGE_KEY=xxxxxxxxxxxxxxxxxxxxxxxx
AZURE_LANGUAGE_ENDPOINT=https://mi-recurso.cognitiveservices.azure.com/
AZURE_REGION=westeurope
```

Y junto a él, el que **sí** se sube:

```bash
# .env.example  -> SÍ se sube, sin valores
AZURE_LANGUAGE_KEY=
AZURE_LANGUAGE_ENDPOINT=
AZURE_REGION=
```

`.env.example` documenta qué variables hacen falta sin revelar ninguna. Es lo que permite que
otra persona, o tú dentro de seis meses, levante el proyecto.

Vamos a crear los dos aquí mismo, con un valor de prueba.

In [ ]:
import pathlib

pathlib.Path(".env").write_text(
    "AZURE_LANGUAGE_KEY=clave-de-prueba-no-real\n"
    "AZURE_LANGUAGE_ENDPOINT=https://mi-recurso.cognitiveservices.azure.com/\n"
    "AZURE_REGION=westeurope\n",
    encoding="utf-8",
)

pathlib.Path(".env.example").write_text(
    "# Copia este fichero a .env y rellena los valores.\n"
    "# El .env no se sube nunca al repositorio.\n"
    "AZURE_LANGUAGE_KEY=\n"
    "AZURE_LANGUAGE_ENDPOINT=\n"
    "AZURE_REGION=westeurope\n",
    encoding="utf-8",
)

pathlib.Path(".gitignore").write_text(
    ".env\n.streamlit/secrets.toml\n__pycache__/\n*.pyc\n",
    encoding="utf-8",
)

print(pathlib.Path(".env.example").read_text(encoding="utf-8"))

### Leerlo

`python-dotenv` carga el `.env` en las variables de entorno del proceso. A partir de ahí se lee
con `os.getenv()`, exactamente igual que en un servidor donde no hay ningún `.env` y las
variables las pone el sistema.

Esa es la gracia: **el código no sabe de dónde sale el valor**.

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

# find_dotenv(usecwd=True) busca el .env desde el directorio actual hacia arriba.
# Es lo que hace que funcione igual en un script y en un cuaderno.
load_dotenv(find_dotenv(usecwd=True))

CLAVE = os.getenv("AZURE_LANGUAGE_KEY")
ENDPOINT = os.getenv("AZURE_LANGUAGE_ENDPOINT")

print("Clave cargada:", bool(CLAVE))
print("Endpoint:", ENDPOINT)

Fíjate en que se ha impreso `bool(CLAVE)` y no la clave. **Nunca imprimas una clave**, ni siquiera
mientras depuras: acaba en la salida guardada del cuaderno, y la salida guardada se entrega.

Si necesitas comprobar que has cargado la correcta, enseña solo los últimos caracteres.

In [ ]:
def pista(secreto, visibles=4):
    """Muestra lo justo para reconocer una clave sin revelarla."""
    if not secreto:
        return "(sin definir)"
    return "..." + secreto[-visibles:]


print("Clave en uso:", pista(CLAVE))

### Fallar pronto y con un mensaje útil

Si falta la variable, el programa debe pararse **ahí**, con un mensaje que diga qué hacer. Lo que
no puede pasar es que la clave sea `None`, llegue a la petición y el servicio responda un 401
que nadie sabe interpretar.

In [ ]:
def lee_variable(nombre):
    """Devuelve el valor de la variable de entorno o explica cómo arreglarlo."""
    valor = os.getenv(nombre)
    if not valor:
        raise RuntimeError(
            f"Falta la variable {nombre}. "
            f"Copia .env.example a .env y rellénala."
        )
    return valor


print(pista(lee_variable("AZURE_LANGUAGE_KEY")))

try:
    lee_variable("AZURE_VISION_KEY")
except RuntimeError as error:
    print("Error esperado:", error)

## 3. Los otros dos sitios: Colab y Streamlit

El `.env` sirve en local. En Colab y en Streamlit Cloud no hay disco propio donde dejarlo, y cada
plataforma tiene su almacén de secretos.

**En Colab**, el panel de la llave en la barra lateral izquierda. Se crea el secreto, se activa
para el cuaderno, y se lee así:

```python
from google.colab import userdata
CLAVE = userdata.get("AZURE_LANGUAGE_KEY")
```

**En Streamlit**, el fichero `.streamlit/secrets.toml`, que tampoco se sube:

```toml
AZURE_LANGUAGE_KEY = "xxxxxxxxxxxx"
AZURE_LANGUAGE_ENDPOINT = "https://mi-recurso.cognitiveservices.azure.com/"
```

En Streamlit Community Cloud ese mismo contenido se pega en el panel de *Secrets* de la
aplicación, y el código no cambia.

### Un solo lector para los tres sitios

Escribe esto una vez y el mismo cuaderno o la misma aplicación funcionan en local, en Colab y
desplegados, sin tocar una línea.

In [ ]:
import os


def obten_secreto(nombre, defecto=None):
    """Lee un secreto de donde esté: Streamlit, Colab o variables de entorno."""
    # 1. Streamlit (local con secrets.toml, o desplegado con el panel de Secrets)
    try:
        import streamlit as st
        if nombre in st.secrets:
            return st.secrets[nombre]
    except Exception:
        pass

    # 2. Google Colab
    try:
        from google.colab import userdata
        valor = userdata.get(nombre)
        if valor:
            return valor
    except Exception:
        pass

    # 3. Variables de entorno, que es lo que carga el .env en local
    return os.getenv(nombre, defecto)


print("Clave:   ", pista(obten_secreto("AZURE_LANGUAGE_KEY")))
print("Endpoint:", obten_secreto("AZURE_LANGUAGE_ENDPOINT"))

Los `except Exception` amplios están justificados aquí y en pocos sitios más: en local no existe
`google.colab`, y fuera de una aplicación Streamlit el acceso a `st.secrets` falla de varias
maneras según la versión. Lo que interesa es seguir probando el siguiente origen, no el detalle
del fallo.

En el código de la aplicación (P2.2 y PR2), esta función vive en `servicios/` y se llama una vez
al arrancar, no en cada petición.

## 4. Comprobar que la clave funciona sin gastar cuota

Antes de montar nada, una llamada mínima al servicio dice si la clave y el endpoint son
correctos. Aquí la llamada es de detección de idioma con un texto de tres palabras: es la
petición más barata que se puede hacer.

La celda siguiente **solo funciona con un recurso de Azure real**. Con la clave de prueba de este
cuaderno dará un fallo de autenticación, que es exactamente lo que queremos ver.

In [ ]:
import requests


def comprueba_credenciales(clave, endpoint, timeout=15):
    """Devuelve (True, mensaje) si la clave y el endpoint funcionan."""
    url = endpoint.rstrip("/") + "/language/:analyze-text?api-version=2022-05-01"
    cabeceras = {"Ocp-Apim-Subscription-Key": clave, "Content-Type": "application/json"}
    cuerpo = {
        "kind": "LanguageDetection",
        "analysisInput": {"documents": [{"id": "1", "text": "Hola"}]},
    }
    try:
        r = requests.post(url, headers=cabeceras, json=cuerpo, timeout=timeout)
    except requests.RequestException as error:
        return False, f"No se pudo contactar: {type(error).__name__}"

    if r.status_code == 200:
        return True, "Credenciales correctas"
    if r.status_code in (401, 403):
        return False, "Clave rechazada. Revisa la clave y que sea la del recurso correcto."
    if r.status_code == 404:
        return False, "Endpoint incorrecto. Comprueba el nombre del recurso y la región."
    return False, f"Respuesta inesperada: HTTP {r.status_code} {r.text[:120]}"


clave = obten_secreto("AZURE_LANGUAGE_KEY")
endpoint = obten_secreto("AZURE_LANGUAGE_ENDPOINT")

if clave and endpoint:
    correcto, mensaje = comprueba_credenciales(clave, endpoint)
    print(("OK: " if correcto else "FALLO: ") + mensaje)
else:
    print("Configura antes AZURE_LANGUAGE_KEY y AZURE_LANGUAGE_ENDPOINT")

Fíjate en que la función distingue el 401 del 404. Son los dos fallos del primer día y se
confunden constantemente:

- **401**: la clave está mal, o es de otro recurso.
- **404**: la clave puede estar bien, pero la URL no apunta a donde crees. Casi siempre es el
  nombre del recurso mal copiado, o el recurso creado en otra región.

## 5. Si se te escapa una clave

Pasa. Lo que se evalúa es la reacción, y el orden importa:

1. **Rotar la clave inmediatamente** en el portal del proveedor. Azure da **dos claves por
   recurso** precisamente para esto: regeneras la primera, la aplicación sigue con la segunda, y
   luego regeneras la segunda.
2. **Después**, limpiar el repositorio. Y si ya estaba subido a un remoto compartido, asumir que
   la clave vieja es pública para siempre.
3. **Revisar el consumo** en el portal por si alguien la ha usado. Un pico de gasto en un
   servicio que no usas es la señal.

El paso 1 es el único que resuelve el problema. Una clave rotada es inofensiva aunque siga
publicada; una clave borrada del último commit pero sin rotar sigue siendo válida.

> En las prácticas de esta unidad, **una clave visible en la entrega es un 0**, y hay que rotarla
> antes de volver a entregar. No es una penalización arbitraria: es el mismo criterio con el que
> se revisaría en cualquier organización.

## 6. Antes de entregar: revísalo

Tres comprobaciones que llevan un minuto y evitan el disgusto.

In [ ]:
import re
import pathlib

# Patrones aproximados de secretos. No detectan todo, pero sí lo habitual.
PATRONES = [
    re.compile(r"[A-Za-z0-9]{32,}"),                       # claves largas sin separadores
    re.compile(r"(?i)(key|token|secret|password)\s*=\s*[\"'][^\"'\s]{8,}"),
]

# En vez de un ejemplo abstracto, revisamos este mismo directorio
SOSPECHOSOS = []
for ruta in pathlib.Path(".").rglob("*"):
    if ruta.is_dir() or ruta.name == ".env" or ".git" in ruta.parts:
        continue
    if ruta.suffix not in (".py", ".ipynb", ".md", ".toml", ".txt", ".yaml", ".yml"):
        continue
    texto = ruta.read_text(encoding="utf-8", errors="ignore")
    for patron in PATRONES:
        for hallazgo in patron.findall(texto):
            SOSPECHOSOS.append((str(ruta), str(hallazgo)[:40]))

if SOSPECHOSOS:
    print("Revisa estos hallazgos antes de entregar:")
    for ruta, hallazgo in SOSPECHOSOS[:15]:
        print(f"  {ruta}: {hallazgo}")
else:
    print("Sin hallazgos evidentes. Repasa igualmente a mano.")

Las otras dos comprobaciones, en el terminal:

```bash
# ¿El .env está realmente ignorado?
git check-ignore -v .env

# ¿Aparece alguna clave en el historial completo?
git log -p | grep -iE "(key|token|secret)\s*=\s*[\"'][^\"']{8,}"
```

Y una que se olvida siempre: **las salidas guardadas del cuaderno**. Si en algún momento
imprimiste la clave, el `.ipynb` la lleva dentro aunque la celda ya no exista. Por eso este
cuaderno solo imprime `pista()` y `bool()`.

## Resumen

| Dónde ejecutas | Dónde va la clave | Cómo se lee |
|---|---|---|
| Local | `.env`, en `.gitignore` | `load_dotenv()` + `os.getenv()` |
| Colab | Panel de secretos (la llave) | `userdata.get()` |
| Streamlit local | `.streamlit/secrets.toml` | `st.secrets[...]` |
| Streamlit Cloud | Panel *Secrets* de la aplicación | `st.secrets[...]` |
| Docker / Azure | Variables de entorno del contenedor | `os.getenv()` |

Y lo que se sube al repositorio en todos los casos: **`.env.example`, y nada más**.

Con esto montado, ya puedes hacer la parte 5 de la P2.1 y pasar al cuaderno **UD2.03**.

In [ ]:
# Limpieza de los ficheros de prueba creados por este cuaderno.
# En tu proyecto real, el .env se queda; aquí no.
for nombre in (".env", ".env.example", ".gitignore"):
    pathlib.Path(nombre).unlink(missing_ok=True)
print("Ficheros de prueba eliminados")